In [19]:
import yfinance as yf
import pandas as pd
from datetime import datetime
from dateutil.relativedelta import relativedelta
from keras import models, metrics, losses, activations, layers, optimizers, callbacks
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, Normalizer
from sklearn.impute import KNNImputer
import numpy as np
import matplotlib as plt
from transformers import pipeline
import requests
import json
import keras_tuner

In [3]:
def get_news_api_key():
    with open('info.json', 'r') as f:
        data = json.load(f)
        return data["news-api-key"]

In [ ]:
pipe = pipeline("text-classification", model="ProsusAI/finbert")
multiplier = { "positive": 1, "neutral": 0, "negative": -1 }

def get_sentiment(keyword, api_key, date=None, start_date=None):
    if date and start_date:
        raise Exception("Date and start_date are mutually exclusive")
    
    headers = {"Authorization": api_key}
    response = requests.get(f"https://newsapi.org/v2/everything?q={keyword}&searchIn=title,description&pageSize=100{"&from=" + start_date if start_date else ""}{"&from=" + date + "&to=" + date if date else ""}", headers=headers)
    response = response.json()

    return response
    # ratings = []
    # sentiments = pipe([article["content"] for article in response["articles"]])
    # for sentiment in sentiments:
    #     ratings.append(multiplier[sentiment["label"]] * sentiment["score"])
    
    # ratings = np.array(ratings)
    # return ratings.mean()

In [ ]:
response = get_sentiment("nvidia", get_news_api_key())

In [ ]:
sources = {}
for name in [article["source"]["name"] for article in response["articles"]]:
    if name in sources:
        sources[name] += 1
    else:
        sources[name] = 1
print(sources)


ratings = []
sentiments = pipe([article["content"] for article in response["articles"]])
for sentiment in sentiments:
    ratings.append(multiplier[sentiment["label"]] * sentiment["score"])
    
ratings = np.array(ratings)
print(ratings.mean())

{'The Verge': 3, 'Gizmodo.com': 4, 'Business Insider': 12, 'Yahoo Entertainment': 40, 'CNET': 1, 'TheStreet': 5, 'Game Revolution': 1, 'Kotaku': 2, 'Fox News': 1, 'heise online': 5, 'Hipertextual': 1, 'Xataka.com': 9, 'Golem.de': 7, 'Gizmodo.jp': 2, 'Windows Central': 1}
-0.02262455573741426


In [46]:
# Download data
start_time = datetime.now() - relativedelta(months=2)
ticker_list = pd.read_csv("s&p500.csv")['Symbol'].dropna().tolist()
data = yf.download(tickers=ticker_list, start=start_time, interval="1d", rounding=True, group_by='ticker', keepna=False)

$BRK.B: possibly delisted; no timezone found
[*******               15%                       ]  77 of 503 completed$K: possibly delisted; no price data found  (1d 2026-01-07 03:48:20.336603 -> 2026-03-07) (Yahoo error = "No data found, symbol may be delisted")
[**********************64%******                 ]  323 of 503 completed$FI: possibly delisted; no price data found  (1d 2026-01-07 03:48:20.336603 -> 2026-03-07) (Yahoo error = "No data found, symbol may be delisted")
[**********************81%**************         ]  407 of 503 completed$IPG: possibly delisted; no price data found  (1d 2026-01-07 03:48:20.336603 -> 2026-03-07) (Yahoo error = "No data found, symbol may be delisted")
[**********************88%*****************      ]  441 of 503 completed$BF.B: possibly delisted; no price data found  (1d 2026-01-07 03:48:20.336603 -> 2026-03-07)
[*********************100%***********************]  501 of 503 completed$WBA: possibly delisted; no price data found  (1d 2026-01-07 0

In [47]:
data2 = data.copy()

In [125]:
data = data2.copy()

In [126]:
# Preprocess data
data.drop(["Adj Close"], level=1, axis=1, inplace=True)

valid_tickers = data.columns.get_level_values(0).unique()

# add change column to every stock
new_cols = {}

for ticker in valid_tickers:
    change = (data[ticker]["Close"]-data[ticker]["Open"])/data[ticker]["Open"]
    new_cols[(ticker, "Change")] = change
    new_cols[(ticker, "RSI")] = (100 - (100/(1 + (change.clip(lower=0).rolling(14, min_periods=2).mean()/-change.clip(upper=0).rolling(14, min_periods=2).mean()))))/100 # as a decimal
    data[(ticker, "5 day MAVG")] = data[ticker]["Close"].rolling(5, min_periods=2).mean()
    data[(ticker, "15 day MAVG")] = data[ticker]["Close"].rolling(15, min_periods=2).mean()

C:\Users\14707\AppData\Local\Temp\ipykernel_21292\4256865121.py:13: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  data[(ticker, "5 day MAVG")] = data[ticker]["Close"].rolling(5, min_periods=2).mean()
C:\Users\14707\AppData\Local\Temp\ipykernel_21292\4256865121.py:14: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  data[(ticker, "15 day MAVG")] = data[ticker]["Close"].rolling(15, min_periods=2).mean()


In [127]:
for ticker in valid_tickers:
    if data[ticker].isna().all(axis=1).any():
        data.drop(ticker, level=0, axis=1, inplace=True)

valid_tickers = data.columns.get_level_values(0).unique()

In [129]:
scaler = MinMaxScaler()
imputer = KNNImputer(n_neighbors=3, weights="distance")

for ticker in valid_tickers:
    data[ticker] = scaler.fit_transform(data[ticker])

# add change column after scaling
for k, v in new_cols.items():
    # make sure not to readd tickers that have been removed
    if k[0] in valid_tickers:
        data[k] = v

# impute after adding RSI back so it sets NaN in RSI to values
for ticker in valid_tickers:
    data[ticker] = imputer.fit_transform(data[ticker])

# # clean data of missing rows and tickers
# for ticker in valid_tickers:
#     last_valid = data[ticker].last_valid_index()
#     # missing all data
#     if last_valid is None or last_valid < pd.Timestamp(datetime.date(datetime.now())):
#         data.drop(ticker, axis=1, level=0, inplace=True)

# # remake data
# valid_tickers = data.columns.get_level_values(0).unique()

C:\Users\14707\AppData\Local\Temp\ipykernel_21292\885803831.py:11: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  data[k] = v


In [119]:
scaler = MinMaxScaler()
imputer = KNNImputer(n_neighbors=3, weights="distance")
for ticker in valid_tickers:
    data[ticker] = imputer.fit_transform(data[ticker])
    data[ticker] = scaler.fit_transform(data[ticker])

# add change column after scaling
for k, v in new_cols.items():
    data[k] = v

In [132]:
# Rearrange data for training
window_size = 10
training_data = []
labels = []
for ticker in valid_tickers:
    for i in range(len(data[ticker])-window_size):
        training_data.append(np.array(data[ticker].iloc[i:i+(window_size-1)]))
        labels.append((data[ticker]["Close"]-data[ticker]["Open"] > 0).iloc[i+window_size])
features = len(data.columns.get_level_values(1).unique())
training_data = np.array(training_data).reshape(-1,(window_size-1),features)
labels = np.array(labels)
X_train, X_test, y_train, y_test = train_test_split(training_data, labels, test_size=0.2, random_state=42, shuffle=True)

In [133]:
print(len(training_data))
print(len(labels))

15376
15376


In [134]:
def build_model(hp):
    model = models.Sequential([
        layers.Input(shape=((window_size-1),features)),
        layers.LSTM(hp.Int("units", min_value=32, max_value=256, step=32), return_sequences=True),
        layers.LSTM(hp.Int("units", min_value=32, max_value=256, step=32)),
        layers.Dense(hp.Int("units", min_value=32, max_value=128, step=32), activation=activations.relu),
        layers.Dense(units=1, activation=activations.sigmoid)
    ])

    model.compile(optimizer=optimizers.Adam(hp.Choice("learning_rate", [1e-1, 1e-2, 1e-3, 1e-4])), loss=losses.binary_crossentropy, metrics=[metrics.binary_accuracy, metrics.FalseNegatives()])
    return model

callback = callbacks.EarlyStopping(monitor="val_loss", patience=16, restore_best_weights=True, verbose=1)
tuner = keras_tuner.Hyperband(
    build_model,
    objective="val_loss",
    max_epochs=128,
    hyperband_iterations=1,
    project_name="trader_optimization_data"
)

tuner.search(X_train, y_train, callbacks=[callback], validation_split=0.2)
tuned = tuner.get_best_models(num_models=3)
model = tuned[0]
model.save("model.keras")
model.fit(X_train, y_train, validation_split=0.2, epochs=128, callbacks=[callback])
model.summary()

Trial 31 Complete [00h 00m 22s]
val_loss: 0.6828472018241882

Best val_loss So Far: 0.6803634762763977
Total elapsed time: 00h 11m 21s


c:\Users\14707\Documents\coding\ml\stock-trader\venv\Lib\site-packages\keras\src\saving\saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 22 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


Epoch 1/128
308/308 ━━━━━━━━━━━━━━━━━━━━ 7s 18ms/step - binary_accuracy: 0.5707 - false_negatives: 1503.0000 - loss: 0.6774 - val_binary_accuracy: 0.5687 - val_false_negatives: 258.0000 - val_loss: 0.6720
Epoch 2/128
308/308 ━━━━━━━━━━━━━━━━━━━━ 5s 16ms/step - binary_accuracy: 0.5840 - false_negatives: 1551.0000 - loss: 0.6746 - val_binary_accuracy: 0.5768 - val_false_negatives: 664.0000 - val_loss: 0.6734
Epoch 3/128
308/308 ━━━━━━━━━━━━━━━━━━━━ 5s 16ms/step - binary_accuracy: 0.5865 - false_negatives: 1726.0000 - loss: 0.6694 - val_binary_accuracy: 0.5789 - val_false_negatives: 485.0000 - val_loss: 0.6703
Epoch 4/128
308/308 ━━━━━━━━━━━━━━━━━━━━ 5s 16ms/step - binary_accuracy: 0.5889 - false_negatives: 1750.0000 - loss: 0.6703 - val_binary_accuracy: 0.5711 - val_false_negatives: 709.0000 - val_loss: 0.6685
Epoch 5/128
308/308 ━━━━━━━━━━━━━━━━━━━━ 5s 16ms/step - binary_accuracy: 0.5922 - false_negatives: 1874.0000 - loss: 0.6669 - val_binary_accuracy: 0.5833 - val_false_negatives: 311

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 9, 96)          │        40,704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 96)             │        74,112 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 96)             │         9,312 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            97 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 372,677 (1.42 MB)

 Trainable params: 124,225 (485.25 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 248,452 (970.52 KB)

In [135]:
model.evaluate(X_test, y_test)

97/97 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - binary_accuracy: 0.6157 - false_negatives: 666.0000 - loss: 0.6511


[0.6511093378067017, 0.6157346963882446, 666.0]